In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
CATALOG = "high_garden"

MARKET_TABLE = f"{CATALOG}.gold.market_metrics"
FORECAST_TABLE = f"{CATALOG}.gold.consumption_forecast"
GROWTH_TABLE = f"{CATALOG}.gold.growth_predictions"
CLUSTER_TABLE = f"{CATALOG}.gold.market_clusters"
ANOMALY_TABLE = f"{CATALOG}.gold.market_anomalies"

OUTPUT_TABLE = f"{CATALOG}.gold.market_opportunities"

In [0]:
market_df = spark.table(
    MARKET_TABLE
)

forecast_df = spark.table(
    FORECAST_TABLE
)

growth_df = spark.table(
    GROWTH_TABLE
)

cluster_df = spark.table(
    CLUSTER_TABLE
)

anomaly_df = spark.table(
    ANOMALY_TABLE
)

print(
    "Market metrics:",
    market_df.count()
)

print(
    "Forecasts:",
    forecast_df.count()
)

print(
    "Growth predictions:",
    growth_df.count()
)

print(
    "Clusters:",
    cluster_df.count()
)

print(
    "Anomaly observations:",
    anomaly_df.count()
)

In [0]:
market_clean = (
    market_df
    .select(
        "country",
        "coffee_type",
        "latest_consumption",
        "cagr_recent",
        "latest_yoy_growth",
        "latest_market_share",
        "volatility_cv",
        "zero_percentage"
    )
)

In [0]:
forecast_clean = (
    forecast_df
    .select(
        "country",
        "coffee_type",

        F.col(
            "crop_year"
        ).alias(
            "forecast_crop_year"
        ),

        F.col(
            "prediction"
        ).alias(
            "forecast_consumption"
        )
    )
)

In [0]:
growth_clean = (
    growth_df
    .select(
        "country",
        "coffee_type",
        "growth_probability",
        "predicted_growth"
    )
)

In [0]:
cluster_clean = (
    cluster_df
    .select(
        "country",
        "coffee_type",
        "cluster",
        "segment"
    )
)

In [0]:
latest_anomaly_year = (
    anomaly_df
    .agg(
        F.max(
            "start_year"
        ).alias(
            "latest_year"
        )
    )
    .first()[
        "latest_year"
    ]
)

recent_anomaly_start = (
    latest_anomaly_year - 4
)

print(
    "Recent anomaly window:",
    recent_anomaly_start,
    "-",
    latest_anomaly_year
)

In [0]:
recent_anomalies = (
    anomaly_df

    .filter(
        F.col(
            "start_year"
        )
        >= recent_anomaly_start
    )

    .groupBy(
        "country",
        "coffee_type"
    )

    .agg(
        F.sum(
            "is_anomaly"
        ).alias(
            "recent_anomaly_count"
        ),

        F.max(
            "anomaly_score"
        ).alias(
            "recent_max_anomaly_score"
        ),

        F.avg(
            "anomaly_score"
        ).alias(
            "recent_avg_anomaly_score"
        )
    )
)

In [0]:
opportunity_df = (
    market_clean

    .join(
        forecast_clean,
        on=[
            "country",
            "coffee_type"
        ],
        how="left"
    )

    .join(
        growth_clean,
        on=[
            "country",
            "coffee_type"
        ],
        how="left"
    )

    .join(
        cluster_clean,
        on=[
            "country",
            "coffee_type"
        ],
        how="left"
    )

    .join(
        recent_anomalies,
        on=[
            "country",
            "coffee_type"
        ],
        how="left"
    )
)

In [0]:
assert (
    opportunity_df.count()
    == 55
), "Expected 55 markets after integration"

In [0]:
duplicate_markets = (
    opportunity_df
    .groupBy(
        "country",
        "coffee_type"
    )
    .count()
    .filter(
        F.col("count") != 1
    )
    .count()
)

assert (
    duplicate_markets == 0
), "Duplicated markets detected after joins"

print(
    "Integrated market table validation passed."
)

In [0]:
null_summary = (
    opportunity_df
    .select(
        [
            F.sum(
                F.col(column)
                .isNull()
                .cast("int")
            ).alias(column)

            for column
            in opportunity_df.columns
        ]
    )
)

display(
    null_summary
)

In [0]:
critical_columns = [
    "forecast_consumption",
    "growth_probability",
    "segment"
]

for column in critical_columns:

    missing = (
        opportunity_df
        .filter(
            F.col(column).isNull()
        )
        .count()
    )

    print(
        column,
        "missing:",
        missing
    )

In [0]:
opportunity_df = (
    opportunity_df

    .withColumn(
        "recent_anomaly_count",
        F.coalesce(
            F.col(
                "recent_anomaly_count"
            ),
            F.lit(0)
        )
    )

    .withColumn(
        "recent_max_anomaly_score",
        F.coalesce(
            F.col(
                "recent_max_anomaly_score"
            ),
            F.lit(0.0)
        )
    )
)

In [0]:
volatility_median = (
    opportunity_df
    .select(
        F.percentile_approx(
            "volatility_cv",
            0.5
        ).alias(
            "median"
        )
    )
    .first()[
        "median"
    ]
)

print(
    "Median volatility:",
    volatility_median
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "volatility_for_score",

        F.coalesce(
            F.col(
                "volatility_cv"
            ),

            F.lit(
                volatility_median
            )
        )
    )
)

In [0]:
def min_max_scale(
    df,
    source_column,
    output_column
):

    bounds = (
        df
        .agg(
            F.min(
                source_column
            ).alias(
                "min_value"
            ),

            F.max(
                source_column
            ).alias(
                "max_value"
            )
        )
        .first()
    )

    min_value = (
        bounds[
            "min_value"
        ]
    )

    max_value = (
        bounds[
            "max_value"
        ]
    )

    if (
        min_value is None
        or
        max_value is None
        or
        max_value == min_value
    ):

        return (
            df
            .withColumn(
                output_column,
                F.lit(0.5)
            )
        )

    return (
        df
        .withColumn(
            output_column,

            (
                F.col(
                    source_column
                )
                -
                F.lit(
                    min_value
                )
            )

            /

            (
                F.lit(
                    max_value
                )
                -
                F.lit(
                    min_value
                )
            )
        )
    )

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "log_forecast_consumption",

        F.log1p(
            F.col(
                "forecast_consumption"
            )
        )
    )
)

In [0]:
opportunity_df = (
    min_max_scale(
        opportunity_df,
        "log_forecast_consumption",
        "demand_score"
    )
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "growth_score",

        F.coalesce(
            F.col(
                "growth_probability"
            ),

            F.lit(0.5)
        )
    )
)

In [0]:
opportunity_df = (
    min_max_scale(
        opportunity_df,
        "volatility_for_score",
        "volatility_normalized"
    )
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "stability_score",

        F.lit(1.0)
        -
        F.col(
            "volatility_normalized"
        )
    )
)

In [0]:
opportunity_df = (
    min_max_scale(
        opportunity_df,
        "recent_max_anomaly_score",
        "anomaly_risk_normalized"
    )
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "anomaly_safety_score",

        F.lit(1.0)
        -
        F.col(
            "anomaly_risk_normalized"
        )
    )
)

In [0]:
DEMAND_WEIGHT = 0.35
GROWTH_WEIGHT = 0.35
STABILITY_WEIGHT = 0.15
ANOMALY_WEIGHT = 0.15

In [0]:
assert abs(
    (
        DEMAND_WEIGHT
        +
        GROWTH_WEIGHT
        +
        STABILITY_WEIGHT
        +
        ANOMALY_WEIGHT
    )
    - 1.0
) < 1e-9

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "opportunity_score_raw",

        F.col(
            "demand_score"
        )
        *
        F.lit(
            DEMAND_WEIGHT
        )

        +

        F.col(
            "growth_score"
        )
        *
        F.lit(
            GROWTH_WEIGHT
        )

        +

        F.col(
            "stability_score"
        )
        *
        F.lit(
            STABILITY_WEIGHT
        )

        +

        F.col(
            "anomaly_safety_score"
        )
        *
        F.lit(
            ANOMALY_WEIGHT
        )
    )
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "opportunity_score",

        F.round(
            F.col(
                "opportunity_score_raw"
            )
            * 100,
            2
        )
    )
)

In [0]:
display(
    opportunity_df
    .select(
        "country",
        "forecast_consumption",
        "growth_probability",
        "demand_score",
        "growth_score",
        "stability_score",
        "anomaly_safety_score",
        "opportunity_score"
    )
    .orderBy(
        F.desc(
            "opportunity_score"
        )
    )
)

In [0]:
percentile_window = (
    Window.orderBy(
        F.col(
            "opportunity_score"
        ).asc()
    )
)

opportunity_df = (
    opportunity_df
    .withColumn(
        "opportunity_percentile",

        F.percent_rank()
        .over(
            percentile_window
        )
    )
)

In [0]:
assert (
    "opportunity_percentile"
    in opportunity_df.columns
)

print(
    "Opportunity percentile created."
)

In [0]:
opportunity_df = (
    opportunity_df
    .withColumn(
        "opportunity_tier",

        F.when(
            F.col(
                "opportunity_percentile"
            ) >= 0.75,

            F.lit(
                "High"
            )
        )

        .when(
            F.col(
                "opportunity_percentile"
            ) >= 0.40,

            F.lit(
                "Medium"
            )
        )

        .otherwise(
            F.lit(
                "Watch"
            )
        )
    )
)

In [0]:
rank_window = (
    Window.orderBy(
        F.col(
            "opportunity_score"
        ).desc()
    )
)

opportunity_df = (
    opportunity_df
    .withColumn(
        "opportunity_rank",

        F.row_number()
        .over(
            rank_window
        )
    )
)

In [0]:
required_ranking_columns = [
    "opportunity_score",
    "opportunity_percentile",
    "opportunity_tier",
    "opportunity_rank"
]

missing_columns = [
    column
    for column
    in required_ranking_columns
    if column
    not in opportunity_df.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

print(
    "Ranking columns created successfully."
)

In [0]:
final_opportunity_df = (
    opportunity_df
    .select(
        "opportunity_rank",

        "country",
        "coffee_type",

        "forecast_crop_year",
        "forecast_consumption",

        "growth_probability",
        "predicted_growth",

        "cagr_recent",
        "latest_yoy_growth",

        "latest_market_share",
        "volatility_cv",

        "cluster",
        "segment",

        "recent_anomaly_count",
        "recent_max_anomaly_score",

        "demand_score",
        "growth_score",
        "stability_score",
        "anomaly_safety_score",

        "opportunity_score",
        "opportunity_percentile",
        "opportunity_tier"
    )
)

In [0]:
display(
    final_opportunity_df
    .orderBy(
        "opportunity_rank"
    )
)

In [0]:
assert (
    final_opportunity_df.count()
    == 55
), "Expected 55 markets"

In [0]:
assert (
    final_opportunity_df
    .filter(
        F.col(
            "opportunity_score"
        ).isNull()
    )
    .count()
    == 0
), "Missing opportunity scores"

In [0]:
assert (
    final_opportunity_df
    .filter(
        F.col(
            "opportunity_rank"
        ).isNull()
    )
    .count()
    == 0
), "Missing rankings"

In [0]:
assert (
    final_opportunity_df
    .filter(
        F.col(
            "opportunity_tier"
        ).isNull()
    )
    .count()
    == 0
), "Missing opportunity tiers"

In [0]:
assert (
    final_opportunity_df
    .filter(
        (
            F.col(
                "opportunity_score"
            ) < 0
        )
        |
        (
            F.col(
                "opportunity_score"
            ) > 100
        )
    )
    .count()
    == 0
), "Opportunity score outside 0-100"

In [0]:
print(
    "Market opportunity validation passed."
)

In [0]:
(
    final_opportunity_df.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        OUTPUT_TABLE
    )
)

In [0]:
persisted_df = spark.table(
    OUTPUT_TABLE
)

print(
    "Persisted markets:",
    persisted_df.count()
)

display(
    persisted_df
    .orderBy(
        "opportunity_rank"
    )
)

In [0]:
%sql

SELECT
    opportunity_rank,
    country,
    coffee_type,

    ROUND(
        forecast_consumption,
        0
    ) AS forecast_consumption,

    ROUND(
        growth_probability * 100,
        1
    ) AS growth_probability_pct,

    segment,

    ROUND(
        volatility_cv,
        3
    ) AS volatility,

    recent_anomaly_count,

    opportunity_score,
    opportunity_tier

FROM high_garden.gold.market_opportunities

ORDER BY opportunity_rank;